# Sortiert die Daten nach der Reihenfolge der Beantwortung der Fragen---> nicht brauchbar

In [ ]:
import pandas as pd

In [ ]:
# CSV laden
df = pd.read_csv('/Users/elias/Desktop/Uni/4. Semester : Masterarbeit/2026-04-02 TYT_answers2.csv', sep=";")

# Überblick verschaffen

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
# Einzigartige Fälle
print("Anzahl Cases:", df["id"].nunique())

In [ ]:
# Einzigartige User
print("Anzahl User:", df["user_id"].nunique())

In [ ]:
# Einträge je User
print(f"Durchschnittliche Einträge pro User: {112184/3339:.2f}")

In [ ]:
# Fehlende Werte finden
df.isna().sum()

# Aufräumen

In [ ]:
# Schauen, ob zwei Spalten redundante Inhalte haben
# bei True sind die Inhalte 1:1 gleich
df["created_at"].equals(df["updated_at"])

In [ ]:
# Schauen, ob die Infos in der Spalte überhaupt relevant sind
df["user_agent"]

In [ ]:
# Schauen, ob die Infos in der Spalte überhaupt relevant sind
df["autosaved"]

In [ ]:
# Spalten "user_agent", "updated_at" und "autosaved" entfernen.
# zudem "notification_fixed" und "notification_date" entfernen, da Miriam sie für unwichtig erklärt hat
df = df.drop(columns=["user_agent", "updated_at", "autosaved", "notification_fixed", "notification_date"])

In [ ]:
df.columns

In [ ]:
# Schauen, dass Werte von Q1-Q8 im erlaubten Bereich sind
# q1 und q8 (0 oder 1)
invalid_q1_q8 = df[
    (df["q1"] < 0) | (df["q1"] > 1) |
    (df["q8"] < 0) | (df["q8"] > 1)
]
print("q1/q8 Fehler:", len(invalid_q1_q8))

# q2 bis q7 (0–10)
invalid_q2_q7 = df[
    (df[["q2","q3","q4","q5","q6","q7"]] < 0).any(axis=1) |
    (df[["q2","q3","q4","q5","q6","q7"]] > 10).any(axis=1)
]
print("q2–q7 Fehler:", len(invalid_q2_q7))

In [ ]:
# Schauen, dass Alter nicht <0 oder >110 ist
df["age"] = pd.to_numeric(df["age"], errors="coerce")
invalid_age = df[(df["age"] < 0) | (df["age"] > 110)]

print(invalid_age[["id", "user_id", "age"]])

In [ ]:
# sicherstellen, dass age numerisch ist
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# Alter und Onset_duration auf NaN setzen
df.loc[(df["user_id"] == 835) & ((df["age"] < 0) | (df["age"] > 110)), "age"] = pd.NA
df.loc[(df["user_id"] == 835), "onset_duration"] = pd.NA

# Kontrolle
df[df["user_id"] == 835]

# Daten aufbereiten

In [ ]:
# Zeitspalten konvertieren
time_cols = ["save_date", "created_at"]

for col in time_cols:
    df[col] = pd.to_datetime(df[col], format="%d.%m.%y %H:%M", errors="coerce")

# Geburtsdatum
date_cols = ["date_of_birth2", "onset_date2"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="%d.%m.%y", errors="coerce")

In [ ]:
df[df["created_at"].isna()]

In [ ]:
df[df["save_date"].isna()]

In [ ]:
# Spalte "save_date" entfernen. Wir werden "created_at" als Zeitstempel nehmen
df = df.drop(columns=["save_date"])

In [ ]:
# Änderungen kontrollieren
df.head()

In [ ]:
import matplotlib.pyplot as plt

# pro User genau ein Eintrag (z. B. erster Eintrag)
df_user = df.drop_duplicates(subset="user_id")

plt.figure(figsize=(20,5))

df_user["age"].value_counts().sort_index().plot(kind="bar")

plt.title("Age Distribution (per User)")
plt.xlabel("Age")
plt.ylabel("Number of Users")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Vorbereitung des Dataframes auf die PM4Py Analyse

In [ ]:
# entweder das
# hier viel mehr Attribute mitgenommen
question_cols = ["q1","q2","q3","q4","q5","q6","q7","q8"]
static_attributes = ["id", "user_id", "created_at", "gender", "age", "sound_env", "user_os"]

df_long = df.melt(
    id_vars=static_attributes,
    value_vars=question_cols,
    var_name="concept:name",
    value_name="value"
)

# Spalten für PM4Py umbenennen
df_long = df_long.rename(columns={
    "id": "case:concept:name",
    "created_at": "time:timestamp"
})

df_long.head()

In [ ]:
# oder das
# hier viel weniger Attribute mitgenommen
'''
question_cols = ["q1","q2","q3","q4","q5","q6","q7","q8"]

df_long = df.melt(
id_vars=["id", "created_at"],
value_vars=question_cols,
var_name="concept:name",
value_name="value"
)

# Spalten für PM4Py umbenennen
df_long = df_long.rename(columns={
"id": "case:concept:name",
"created_at": "time:timestamp"
})

df_long.head()
'''

In [ ]:
# Sortieren
df_long["order"] = df_long["concept:name"].str.extract("(\d+)").astype(int)
df_long = df_long.sort_values(by=["case:concept:name", "order"])


# künstliche Zeitstempel erzeugen
df_long["time:timestamp"] += pd.to_timedelta(
    df_long.groupby("case:concept:name").cumcount(), unit="s"
)

In [ ]:
df_long

In [ ]:
# Aggregieren
df_agg = df.groupby("id").agg({"age": "first"}).reset_index()

df_long = df_long.merge(
    df_agg,
    left_on="case:concept:name",
    right_on="id",
    how="left"
)

In [ ]:
df_long[df_long["time:timestamp"].isna()]

In [ ]:
'''# evtl. jetzt noch unvollständige Werte entfernen
df_long = df_long.dropna(subset=[
    "case:concept:name",
    "concept:name",
    "time:timestamp"
])'''

In [ ]:
df_long["case:concept:name"] = df_long["case:concept:name"].astype(str)
df_long["concept:name"] = df_long["concept:name"].astype(str)

# Kontrollieren
print(df_long["case:concept:name"].dtype)
print(df_long["concept:name"].dtype)

In [ ]:
# Kontrollieren, dass jetzt die Sortierung nach Case und Question sortiert ist
df_long

In [ ]:
df_long[df_long["value"].isna()]

In [ ]:
import pm4py

log = pm4py.convert_to_event_log(df_long)

print("Anzahl Traces:", len(log))

In [ ]:
# euren gewünschten Pfad hernehmen. Dann auch hernehmen, um damit weiter zu arbeiten

#pm4py.write_xes(log, '/Users/elias/Desktop/Uni/4. Semester : Masterarbeit/2026-04-06 TYT_event_log.xes')